# 4.15 · 稳健回归 / Robust Regression

> **课程定位 / Where this fits**
> **Part 4 第 15 课**。4.2 诊断出异常/影响点, 3.3 讲过"真实异常值别删"。但**异常值确实会毁掉 OLS**（平方损失放大大误差）。稳健回归 = **改损失/算法, 让异常值少出声**: Huber / RANSAC / Theil-Sen。这接续 2.1 的稳健统计、4.14 的中位数思想。
> Outliers wreck OLS (squared loss amplifies them). Robust regression changes the loss/algorithm so outliers carry less weight.

> 💡 **面试相关 / Interview-relevant**
> - "异常值怎么影响线性回归" ★★★★（平方损失放大）
> - "Huber loss 是什么" ★★★★（小误差平方/大误差线性）
> - "RANSAC 原理" ★★★（随机采样找内点）
> - "稳健回归 vs 删异常值" ★★★

---

## 学习目标 / Learning Objectives
1. 量化**单个异常值**如何毁掉 OLS（平方损失的放大效应）。
2. 掌握三种稳健方法的思想：**Huber**(混合损失) / **RANSAC**(共识采样) / **Theil-Sen**(中位斜率)。
3. 理解**击穿点**（2.1 节）在回归里的含义。
4. 知道何时用稳健回归 vs 删异常值(3.3) vs 分位数(4.14)。

## 目录 / TOC
1. [异常值如何毁掉 OLS ⭐](#1)
2. [Huber: 混合损失 ⭐](#2)
3. [RANSAC: 共识采样 ⭐](#3)
4. [Theil-Sen: 中位斜率](#4)
5. [四方法对决 + 击穿点](#5)
6. [选择决策](#6)
7. [小结](#7)


<a id="1"></a>
## 1. 异常值如何毁掉 OLS ⭐ / How Outliers Wreck OLS

OLS 最小化**平方**误差 $\sum(y-\hat{y})^2$。问题: **平方放大大误差**——一个远离的点产生巨大的平方误差, OLS 为了减小它会**大幅偏转整条回归线**去迁就这个异常点。

这是 2.1 节"均值不稳健"（击穿点 0）在回归里的体现: **OLS 的击穿点也是 0**——理论上**一个**足够极端的点就能把回归线拉到任意位置。
This is the regression version of 2.1's "mean has breakdown point 0" — one extreme point can drag the OLS line anywhere.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

# 干净的线性数据 + 几个 y 方向异常值 / clean line + a few y-outliers
n = 100
x = np.sort(rng.uniform(0, 10, n))
y = 2*x + 1 + rng.normal(0, 1, n)
# 注入 5 个异常值 (高杠杆+大残差) / inject 5 outliers
y_dirty = y.copy()
y_dirty[-5:] = y[-5:] - 40        # 把右端 5 个点拉到很低
X = x.reshape(-1, 1)

ols_clean = LinearRegression().fit(X, y)
ols_dirty = LinearRegression().fit(X, y_dirty)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(x[:-5], y_dirty[:-5], alpha=0.4, s=15, label="正常点")
ax.scatter(x[-5:], y_dirty[-5:], c="red", s=60, label="异常值(5个)")
xp = np.linspace(0, 10, 50).reshape(-1, 1)
ax.plot(xp, ols_clean.predict(xp), "g--", lw=2, label=f"OLS(无异常) 斜率={ols_clean.coef_[0]:.2f}")
ax.plot(xp, ols_dirty.predict(xp), "r-", lw=2, label=f"OLS(含异常) 斜率={ols_dirty.coef_[0]:.2f}")
ax.legend(fontsize=9); ax.set_title("5 个异常值(5%)就把 OLS 斜率从 2.0 拉偏")
plt.tight_layout(); plt.show()
print(f"真实斜率 2.0; 无异常 OLS={ols_clean.coef_[0]:.2f}; 含 5% 异常 OLS={ols_dirty.coef_[0]:.2f}")
print("→ 仅 5% 异常值就显著拉偏回归线 — 平方损失对大误差过度敏感")


<a id="2"></a>
## 2. Huber: 混合损失 ⭐ / Huber Loss

**Huber 回归**: 用一个**混合损失**——小误差用平方（保持高效率）, 大误差用线性（限制异常值的影响）：

$$L_\delta(r) = \begin{cases} \frac{1}{2}r^2 & |r| \le \delta \;(\text{小误差: 平方})\\ \delta(|r| - \frac{1}{2}\delta) & |r| > \delta\;(\text{大误差: 线性}) \end{cases}$$

**直觉**: $\delta$ 是"异常值阈值"。误差小于 $\delta$ 时和 OLS 一样（不损失效率）; 超过 $\delta$ 后惩罚只**线性**增长（不像平方那样爆炸）→ **异常值被"封顶"了影响力**。这正是 4.9 SVR ε-不敏感、4.14 pinball 的同族思想: **改损失函数限制大误差**。
Huber: squared for small errors (efficient), linear for large (caps outlier influence) — delta is the outlier threshold.


In [ ]:
from sklearn.linear_model import HuberRegressor

# Huber loss 几何 / Huber loss geometry
r = np.linspace(-4, 4, 300)
delta = 1.0
huber = np.where(np.abs(r) <= delta, 0.5*r**2, delta*(np.abs(r)-0.5*delta))
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(r, 0.5*r**2, "r--", label="平方损失(OLS)")
axes[0].plot(r, huber, "b-", lw=2, label=f"Huber (δ={delta})")
axes[0].plot(r, np.abs(r), "g:", label="绝对损失")
axes[0].axvspan(-delta, delta, alpha=0.1, color="blue")
axes[0].legend(); axes[0].set_xlabel("残差 r"); axes[0].set_title("Huber: |r|<δ 平方, |r|>δ 线性")

# Huber vs OLS 拟合 / Huber vs OLS fit
huber_reg = HuberRegressor().fit(X, y_dirty)
axes[1].scatter(x[:-5], y_dirty[:-5], alpha=0.4, s=15)
axes[1].scatter(x[-5:], y_dirty[-5:], c="red", s=50)
xp = np.linspace(0,10,50).reshape(-1,1)
axes[1].plot(xp, ols_dirty.predict(xp), "r-", lw=2, label=f"OLS 斜率={ols_dirty.coef_[0]:.2f}")
axes[1].plot(xp, huber_reg.predict(xp), "b-", lw=2, label=f"Huber 斜率={huber_reg.coef_[0]:.2f}")
axes[1].plot(xp, ols_clean.predict(xp), "g--", lw=1.5, label="真实(无异常)")
axes[1].legend(fontsize=8); axes[1].set_title("Huber 几乎不受异常值影响")
plt.tight_layout(); plt.show()
print(f"含异常时: OLS 斜率={ols_dirty.coef_[0]:.2f}, Huber 斜率={huber_reg.coef_[0]:.2f} (真实 2.0)")
print("Huber 把异常值的大残差用线性惩罚 → 它们拉不动回归线 → 斜率接近真实")


<a id="3"></a>
## 3. RANSAC: 共识采样 ⭐ / RANSAC

**RANSAC (RANdom SAmple Consensus)** 思路完全不同——不改损失, 而是**反复随机采样小子集拟合, 找内点最多的那个模型**：

```
重复多次:
  1. 随机选最少的点拟合一条线 (2 点定一线)
  2. 看有多少其他点落在这条线附近 (内点 inliers)
  3. 记下内点最多的模型
最后: 用所有内点重新拟合
```

**威力**: RANSAC 能容忍**高达 50% 甚至更多**的异常值（击穿点高）——因为它**完全无视外点(outliers)**, 只要能采样到一个"干净"的子集。计算机视觉里拟合（如图像配准）的标配。
RANSAC ignores outliers entirely by finding the model with the most inliers — tolerates very high contamination.


In [ ]:
from sklearn.linear_model import RANSACRegressor

# 极端污染: 40% 异常值 / extreme 40% contamination
y_extreme = y.copy()
out_idx = rng.choice(n, 40, replace=False)
y_extreme[out_idx] = rng.uniform(-30, 50, 40)    # 40% 完全随机的异常

ransac = RANSACRegressor(random_state=0).fit(X, y_extreme)
ols_ext = LinearRegression().fit(X, y_extreme)
inlier_mask = ransac.inlier_mask_

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(x[inlier_mask], y_extreme[inlier_mask], alpha=0.5, s=15, label="RANSAC 识别的内点")
ax.scatter(x[~inlier_mask], y_extreme[~inlier_mask], c="red", alpha=0.5, s=15, label="RANSAC 识别的外点")
xp = np.linspace(0,10,50).reshape(-1,1)
ax.plot(xp, ols_ext.predict(xp), "r-", lw=2, label=f"OLS 斜率={ols_ext.coef_[0]:.2f}")
ax.plot(xp, ransac.predict(xp), "b-", lw=2, label=f"RANSAC 斜率={ransac.estimator_.coef_[0]:.2f}")
ax.legend(fontsize=8); ax.set_title("40% 异常值! OLS 崩溃, RANSAC 自动识别内点并准确拟合")
plt.tight_layout(); plt.show()
print(f"40% 异常下: OLS 斜率={ols_ext.coef_[0]:.2f} (崩), RANSAC={ransac.estimator_.coef_[0]:.2f} (真实 2.0!)")
print(f"RANSAC 自动把 {(~inlier_mask).sum()} 个点判为外点并忽略 → 高击穿点")


<a id="4"></a>
## 4. Theil-Sen: 中位斜率 / Theil-Sen Estimator

**Theil-Sen**: 最优雅的稳健回归——**取所有点对连线斜率的中位数**作为回归斜率：

$$\hat{\beta} = \text{median}\Big\{\frac{y_j - y_i}{x_j - x_i} : i < j\Big\}$$

**直觉**: 中位数天生稳健（2.1 节, 击穿点 ~29%）。异常点产生的极端斜率会落在两端, 被中位数过滤掉。**简单、无超参、有理论保证**, 但 $O(n^2)$ 点对, 大数据慢。
Theil-Sen takes the median of all pairwise slopes — inherently robust via the median's high breakdown point.


In [ ]:
from sklearn.linear_model import TheilSenRegressor

theil = TheilSenRegressor(random_state=0).fit(X, y_dirty)
print(f"5% 异常下各方法斜率 (真实 2.0):")
print(f"  OLS:       {ols_dirty.coef_[0]:.3f}  (被拉偏)")
print(f"  Huber:     {HuberRegressor().fit(X, y_dirty).coef_[0]:.3f}")
print(f"  Theil-Sen: {theil.coef_[0]:.3f}")
print(f"  RANSAC:    {RANSACRegressor(random_state=0).fit(X, y_dirty).estimator_.coef_[0]:.3f}")
print("\nTheil-Sen = 所有点对斜率的中位数; 异常点对的极端斜率被中位数过滤")


<a id="5"></a>
## 5. 四方法对决 + 击穿点 / Showdown & Breakdown Points

**击穿点** (2.1 节): 要让估计器任意坏需要多大比例的污染。回归版：


In [ ]:
# 不同污染比例下各方法的稳健性 / robustness across contamination levels
methods = {
    "OLS": LinearRegression(),
    "Huber": HuberRegressor(),
    "Theil-Sen": TheilSenRegressor(random_state=0),
    "RANSAC": RANSACRegressor(random_state=0),
}
contam_levels = [0.0, 0.1, 0.2, 0.35, 0.5]
print(f"{'方法':<12} " + " ".join(f"{int(c*100):>5}%" for c in contam_levels) + "   (估计斜率, 真实=2.0)")
print("-" * 55)
for name, model in methods.items():
    row = []
    for c in contam_levels:
        yd = y.copy()
        if c > 0:
            idx = rng.choice(n, int(n*c), replace=False)
            yd[idx] = rng.uniform(-30, 50, len(idx))
        try:
            model.fit(X, yd)
            slope = model.estimator_.coef_[0] if name=="RANSAC" else model.coef_[0]
            row.append(f"{slope:>6.2f}")
        except: row.append("  fail")
    print(f"{name:<12} " + " ".join(row))
print("\nOLS: 一有污染就偏; Huber: 中度污染稳; Theil-Sen: ~29% 击穿点; RANSAC: 最高(~50%+)")
print("击穿点排序: OLS(0) < Huber < Theil-Sen(~29%) < RANSAC(可>50%)")


<a id="6"></a>
## 6. 选择决策 / Decision Guide

```
先问: 异常值是错误还是真实? (3.3)
  数据错误 → 先修正/删除 (3.3), 别直接上稳健回归掩盖问题
  真实罕见值 → 稳健回归 (不该删)

选哪个稳健方法:
  少量异常(<10%) + 要效率   → Huber (接近 OLS 但抗异常)
  中度异常 + 简单无超参      → Theil-Sen (中位斜率, 但 O(n²) 慢)
  大量异常(可达50%) + 找内点 → RANSAC (CV/几何拟合标配)
  异方差(非异常)但要区间     → 分位数回归 (4.14, 不同问题)

也可考虑: 树模型(4.11-13)天然较抗异常值(基于排序切分, 不放大)
```

⚠ **稳健回归不是万能药**: 它降低异常值影响, 但如果异常值其实**携带信息**（如欺诈), 你想要的是**单独建模**它们（异常检测, 3.3/Part 5.14）, 而非压制。
Robust regression suppresses outliers; if outliers are informative (fraud), model them separately instead.


<a id="7"></a>
## 7. 小结 / Summary

```
OLS 击穿点=0: 平方损失放大大误差, 一个极端点能拉偏整条线
稳健三法:
  Huber ⭐ — 混合损失(小误差平方/大误差线性), δ=阈值, 效率高
  RANSAC ⭐ — 随机采样找内点最多的模型, 击穿点最高(~50%+)
  Theil-Sen — 所有点对斜率的中位数, 无超参但 O(n²)
击穿点: OLS(0) < Huber < Theil-Sen(~29%) < RANSAC
先分辨异常值身份(3.3): 错误→删, 真实→稳健回归, 信息→单独建模
树模型天然较抗异常(基于排序); 分位数回归(4.14)是另一问题(异方差)
```

### 💡 面试速查
1. **OLS 不稳健**: 平方损失, 击穿点 0, 一个异常点拉偏
2. **Huber**: 小误差平方(效率) + 大误差线性(限制异常)
3. **RANSAC**: 采样找内点最多模型, 容忍高达 50% 异常
4. **Theil-Sen**: 点对斜率中位数 (中位数稳健 2.1)
5. **稳健 vs 删除**: 真实异常用稳健, 数据错误先删(3.3)

### 下一节
**4.16 等张回归**——Part 4 收官。前面都假设特定函数形式。等张回归只假设"**单调**"(y 随 x 不减), 无参数, 主要用于**概率校准**(Part 5.8) 和剂量-反应曲线。
